
# LV geometry, AHA segmentation, displacement, and strain visualisation

This notebook merges the duplicated logic from the two original notebooks and keeps all requested outputs in one pipeline.

## What this notebook produces

1. **Complete LV geometry** (no colouring)
2. **AHA17 segmentation view** (neutral surface, 17 partitions visible, no fill colouring)
3. **AHA5 grouped view** (AHA5 fill colours, with both AHA17 and AHA5 boundaries)
4. **Overall end-diastolic displacement** (no scalar colouring; original vs displaced overlay)
5. **Full-field strain visualisations** for 3 directions: circumferential, radial, longitudinal
6. **AHA5-grouped strain visualisations** for the same 3 directions
7. **Lateral-group subsegment strain visualisations** for the same 3 directions
8. **Lateral-group strain distribution plots** (KDE grid and boxplots)

## Important grouping choice

This notebook **keeps the current `AHA17_ELEM_FIELDS` unchanged** and **rewrites the AHA5 grouping accordingly**:

- **Anterior**: segments **3, 9**
- **Lateral**: segments **4, 5, 10, 11**
- **Inferior**: segments **6, 12**
- **Septal**: segments **1, 2, 7, 8**
- **Apical**: segments **13, 14, 15, 16, 17**

So the **Lateral group** used throughout this notebook is:

```python
LATERAL_SEGMENTS = (4, 5, 10, 11)
```



## Environment

If `pyvista` / `vtk` are not installed in your current notebook kernel, run the next setup cell once.


In [1]:

import importlib
import subprocess
import sys

REQUIRED = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'scipy': 'scipy',
    'pyvista': 'pyvista',
    'vtk': 'vtk',
}

missing = []
for mod, pkg in REQUIRED.items():
    try:
        importlib.import_module(mod)
    except Exception:
        missing.append(pkg)

if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
else:
    print('All required packages are already available.')


All required packages are already available.


In [2]:

from pathlib import Path
import warnings

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap, Normalize
from scipy.io import loadmat
from scipy.stats import gaussian_kde
import pyvista as pv

warnings.filterwarnings('ignore', category=FutureWarning)

try:
    pv.start_xvfb()
except Exception:
    pass

pv.global_theme.background = 'white'
pv.global_theme.window_size = [1600, 800]

print(f'PyVista: {pv.__version__}')


PyVista: 0.47.3


C:\Users\r4718\AppData\Local\Temp\ipykernel_15484\2958153053.py:16: PyVistaDeprecationWarning: This function is deprecated and will be removed in future version of PyVista. Use vtk with osmesa instead.
  pv.start_xvfb()



## Configuration

Adjust the paths or rendering parameters here if needed.


In [3]:
# ================================
# 1) User configuration
# ================================

def resolve_existing_path(name: str) -> Path:
    candidates = [
        Path.cwd() / name,
        Path('/mnt/data') / name,
        Path(name),
    ]
    for p in candidates:
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'Cannot find file: {name}')

INP_PATH   = resolve_existing_path('hexheartLVmeshF60S45.inp')
AHA_MAT    = resolve_existing_path('AHALVMeshDivisionPCAReconstructed.mat')
STRAIN_MAT = resolve_existing_path('strainComparison_Baseline.mat')

OUTPUT_DIR = Path('merged_lv_visualization_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_VTU = True

WINDOW_SIZE   = (1600, 800)
WINDOW_SIZE_3 = (2100, 800)     # for 3-panel figures (anterior / posterior / short-axis)
DPI           = 300
ZOOM          = 1.08
PARALLEL_PROJ = True
CLIP_NORMAL   = (1.0, 0.0, 0.0)

# --- Surface appearance --------------------------------------------------
DEFAULT_SURFACE_COLOR = '#E8E8E8'     # [P0] lightened so the AHA17 lines can bite
WIRE_COLOR = '#FFFFFF'
WIRE_ALPHA = 0.18
WIRE_WIDTH = 0.35

OUTLINE_COLOR = '#000000'
OUTLINE_WIDTH = 2.6

# [P0] AHA17 edges were #8A8A8A / 0.95 — invisible at 300 DPI on a neutral surface.
AHA17_EDGE_COLOR = '#1A1A1A'
AHA17_EDGE_WIDTH = 2.2
AHA17_EDGE_ALPHA = 0.98

AHA5_EDGE_COLOR = '#000000'
AHA5_EDGE_WIDTH = 3.2
AHA5_EDGE_ALPHA = 1.00

DIV_CMAP = plt.get_cmap('RdBu_r')

# [P3] Strain colour-limit controls. The basal annular rim (exposed cut face)
# carries genuine but visually dominant extremes; we exclude a thin band of
# elements near the basal plane when computing clim so that the wall body
# actually shows usable contrast. Rendering still colours every cell.
STRAIN_CLIM_PCT       = (5.0, 95.0)
STRAIN_BASAL_EXCLUDE  = 0.05   # top 5% of the z-extent is treated as rim

# [P2] Strain-tensor-column contract  ------------------------------------
# `strain_tensor_cra` has 6 columns. This notebook assumes the first three are
# the diagonal (normal) strains in the circumferential / radial / longitudinal
# basis, in that order. Signs observed on this dataset
#     col0 mean +0.119  (circ stretch at ED: expected +)
#     col1 mean -0.131  (radial thinning at ED: expected -)
#     col2 mean +0.156  (longitudinal stretch at ED: expected +)
# are consistent with that interpretation.
#
# HOWEVER: the 1D fields `radialStrain_cra`, `longiStrain_cra`,
# `fiberStrain_cra` in the same MAT file are *not* sign-flipped copies of
# columns 0–2 (linear fit slopes are ~1.7-2.0, not ±1). They were computed by
# a different pipeline and cannot be used as a cross-check. If you ever find
# strain plots disagree with your MATLAB output, return to the MATLAB source
# that produced `strain_tensor_cra` and verify the Voigt ordering; then adjust
# STRAIN_COLS below.
STRAIN_COLS = {
    'circ': 0,   # verify: column holding E_cc
    'rad':  1,   # verify: column holding E_rr
    'lon':  2,   # verify: column holding E_ll
}



## Label definitions and remapped AHA5 grouping


In [4]:

# ================================
# 2) AHA label definitions
# ================================

# Keep the existing AHA17 field mapping unchanged.
AHA17_ELEM_FIELDS = [
    ('elem_basa_InfSept', 1), ('elem_basa_AntSept', 2), ('elem_basa_Ant',    3),
    ('elem_base_AntLat',  4), ('elem_base_InfLat',  5), ('elem_base_Inf',    6),
    ('elem_midd_InfSept', 7), ('elem_midd_AntSept', 8), ('elem_midd_Ant',    9),
    ('elem_midd_AntLat', 10), ('elem_midd_InfLat', 11), ('elem_midd_Inf',   12),
    ('elem_apex_Sept',   13), ('elem_apex_Ant',    14), ('elem_apex_Lat',   15),
    ('elem_apex_Inf',    16), ('elem_apicalRegion',17),
]

AHA17_LABELS = {
    1: 'Basal inferoseptal',
    2: 'Basal anteroseptal',
    3: 'Basal anterior',
    4: 'Basal anterolateral',
    5: 'Basal inferolateral',
    6: 'Basal inferior',
    7: 'Mid inferoseptal',
    8: 'Mid anteroseptal',
    9: 'Mid anterior',
    10: 'Mid anterolateral',
    11: 'Mid inferolateral',
    12: 'Mid inferior',
    13: 'Apical septal',
    14: 'Apical anterior',
    15: 'Apical lateral',
    16: 'Apical inferior',
    17: 'Apical cap / conflict',
}

# Remapped AHA5 grouping based on the preserved AHA17 numbering above.
AHA17_TO_AHA5 = {
    3: 1, 9: 1,                       # Anterior
    4: 2, 5: 2, 10: 2, 11: 2,         # Lateral
    6: 3, 12: 3,                      # Inferior
    1: 4, 2: 4, 7: 4, 8: 4,           # Septal
    13: 5, 14: 5, 15: 5, 16: 5, 17: 5 # Apical
}

AHA5_GROUP_NAMES = {
    1: 'Anterior',
    2: 'Lateral',
    3: 'Inferior',
    4: 'Septal',
    5: 'Apical',
}

AHA5_COLORS = [
    '#0072B2',  # Anterior
    '#009E73',  # Lateral
    '#E69F00',  # Inferior
    '#CC79A7',  # Septal
    '#D55E00',  # Apical
]
AHA5_CMAP = ListedColormap(AHA5_COLORS, name='aha5')

LATERAL_SEGMENTS = (4, 5, 10, 11)
LATERAL_SUB_COLORS = ['#004D40', '#1B9E77', '#4DB6AC', '#80CBC4']
LATERAL_SUB_CMAP = ListedColormap(LATERAL_SUB_COLORS, name='lateral4')

STRAIN_LABELS = {
    'circ': 'Circumferential strain',
    'rad': 'Radial strain',
    'lon': 'Longitudinal strain',
}



## I/O and mesh construction helpers


In [5]:

# ================================
# 3) I/O helpers
# ================================

def _as_1d_int(x):
    a = np.asarray(x).reshape(-1)
    if a.size and np.issubdtype(a.dtype, np.number):
        a = a[np.isfinite(a)]
    return a.astype(np.int64)


def parse_abaqus_inp(inp_file):
    """Parse *Node and *Element(type=C3D8/C3D8H) from an Abaqus .inp file."""
    node_ids, points, conn = [], [], []
    in_nodes = False
    in_hex = False

    with open(inp_file, 'r', encoding='utf-8', errors='ignore') as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith('**'):
                continue

            if line.startswith('*'):
                up = line.upper()
                in_nodes = up.startswith('*NODE')
                is_elem = up.startswith('*ELEMENT')
                in_hex = is_elem and ('TYPE=C3D8' in up or 'TYPE=C3D8H' in up)
                continue

            parts = [p.strip() for p in line.split(',') if p.strip()]
            if in_nodes and len(parts) >= 4:
                node_ids.append(int(parts[0]))
                points.append([float(parts[1]), float(parts[2]), float(parts[3])])
            elif in_hex and len(parts) >= 9:
                conn.append([int(x) for x in parts[1:9]])

    if not node_ids or not conn:
        raise RuntimeError('Failed to parse nodes or hex elements from the INP file.')

    node_ids = np.asarray(node_ids, dtype=np.int64)
    points = np.asarray(points, dtype=np.float64)
    conn_1b = np.asarray(conn, dtype=np.int64)

    lookup = np.full(node_ids.max() + 2, -1, dtype=np.int64)
    lookup[node_ids] = np.arange(node_ids.size)
    conn_0b = lookup[conn_1b]
    if (conn_0b < 0).any():
        raise RuntimeError('Some elements reference undefined node IDs.')

    return points, conn_0b


def load_aha_labels(mat_file, n_cells):
    """Load AHA17 labels from the MATLAB struct and remap them to AHA5."""
    mat = loadmat(mat_file, squeeze_me=True, struct_as_record=False)
    if 'AHALVMeshDivision' not in mat:
        raise KeyError("Missing 'AHALVMeshDivision' in the AHA MATLAB file.")

    s = mat['AHALVMeshDivision']

    def _get(name):
        if hasattr(s, name):
            return getattr(s, name)
        return None

    node_regions = _as_1d_int(_get('nodeRegions'))
    if node_regions.size == 0:
        raise ValueError('Missing nodeRegions in the AHA MATLAB file.')

    aha17 = np.zeros(n_cells, dtype=np.int32)
    hit_count = np.zeros(n_cells, dtype=np.int32)
    missing_fields = []

    for field, rid in AHA17_ELEM_FIELDS:
        ids = _get(field)
        if ids is None:
            missing_fields.append(field)
            continue
        idx = np.unique(_as_1d_int(ids) - 1)
        if (idx < 0).any() or (idx >= n_cells).any():
            raise ValueError(f'{field}: element IDs are out of range.')
        first_hit = hit_count[idx] == 0
        aha17[idx[first_hit]] = rid
        hit_count[idx] += 1

    if missing_fields:
        raise ValueError(f'Missing MATLAB fields: {missing_fields}')
    if (hit_count == 0).any():
        raise ValueError(f'{int((hit_count == 0).sum())} elements are not covered by any elem_* field.')

    conflict_mask = hit_count > 1
    aha17[conflict_mask] = 17

    aha5 = np.array([AHA17_TO_AHA5[int(x)] for x in aha17], dtype=np.int32)
    return node_regions, aha17, aha5, conflict_mask


def load_strain_struct(mat_file, n_cells_expected=None, n_points_expected=None):
    """Load strain tensor and nodal displacement from strainComparison_Baseline.mat."""
    mat = loadmat(mat_file, squeeze_me=True, struct_as_record=False)
    if 'strainComparison' not in mat:
        raise KeyError("Missing 'strainComparison' in the strain MATLAB file.")

    s = mat['strainComparison']
    T = np.asarray(getattr(s, 'strain_tensor_cra'), dtype=np.float64)
    dis = np.asarray(getattr(s, 'dis'), dtype=np.float64)
    node_ori = np.asarray(getattr(s, 'node_ori'), dtype=np.float64)

    if n_cells_expected is not None and T.shape[0] != n_cells_expected:
        raise ValueError('strain_tensor_cra does not match the mesh cell count.')
    if n_points_expected is not None and dis.shape[0] != n_points_expected:
        raise ValueError('dis does not match the mesh point count.')
    if n_points_expected is not None and node_ori.shape[0] != n_points_expected:
        raise ValueError('node_ori does not match the mesh point count.')

    return {
        'strain_tensor': T,
        'dis': dis,
        'node_ori': node_ori,
    }


def build_grid(points, conn, node_regions, aha17, aha5, strain_tensor=None, dis=None):
    n_cells = conn.shape[0]
    cells = np.hstack([np.full((n_cells, 1), 8, dtype=np.int64), conn]).ravel()
    celltypes = np.full(n_cells, pv.CellType.HEXAHEDRON, dtype=np.uint8)
    grid = pv.UnstructuredGrid(cells, celltypes, points)

    grid.cell_data['AHA17'] = aha17.astype(np.int32)
    grid.cell_data['AHA5'] = aha5.astype(np.int32)

    # Precomputed RGB so rendering bypasses VTK's IndexedLookup, which otherwise
    # remaps colours when a sub-mesh (e.g. a short-axis slab) is missing some
    # AHA5 categories. See the "second patch" note at the top of this notebook.
    rgb = np.zeros((aha5.size, 3), dtype=np.float32)
    palette_rgb = np.array([
        [float(int(c[1:3], 16)), float(int(c[3:5], 16)), float(int(c[5:7], 16))]
        for c in AHA5_COLORS
    ], dtype=np.float32) / 255.0
    for gid in range(1, 6):
        m = aha5 == gid
        rgb[m] = palette_rgb[gid - 1]
    grid.cell_data['AHA5_rgb'] = rgb

    if node_regions.size == points.shape[0]:
        grid.point_data['node_regions'] = node_regions.astype(np.int32)

    if strain_tensor is not None:
        for key, idx in STRAIN_COLS.items():
            grid.cell_data[f'strain_{key}'] = strain_tensor[:, idx].astype(np.float64)

    if dis is not None:
        grid.point_data['disp_x'] = dis[:, 0]
        grid.point_data['disp_y'] = dis[:, 1]
        grid.point_data['disp_z'] = dis[:, 2]
        grid.point_data['disp_mag'] = np.linalg.norm(dis, axis=1)

    return grid


def fix_septal_inferior_boundary(aha17, aha5, cell_centers,
                                  inf_theta_min=30.0, inf_theta_max=90.0,
                                  verbose=True):
    """Data-cleaning step for the Septal-Inferior boundary region.

    The source MAT file contains a non-trivial number of cells whose AHA17
    label places them in the Anterior (3, 9) or Lateral (4, 5, 10, 11)
    segments, but whose actual spatial position in the short-axis (x, y)
    plane falls inside the Inferior angular band occupied by seg 6 / 12.
    These cells are mislabelled (e.g. seg 4/10 cells at theta ~ +81 deg,
    which is anatomically opposite to AntLat's correct ~ -45 deg).

    This function identifies such cells by angular position and reassigns
    them to the Inferior segment — seg 6 if the cell sits in the basal
    half-slab (current AHA17 in 1..6) and seg 12 if in the mid half-slab
    (current AHA17 in 7..12). AHA5 is updated to Inferior (3) accordingly.
    Apical cells (AHA17 in 13..17) are left untouched.

    Returns (aha17_fixed, aha5_fixed, n_reassigned).
    """
    cc = np.asarray(cell_centers)
    cx, cy = cc[:, 0].mean(), cc[:, 1].mean()
    theta = np.degrees(np.arctan2(cc[:, 1] - cy, cc[:, 0] - cx))

    is_basal_mid = (aha17 >= 1) & (aha17 <= 12)
    is_basal     = (aha17 >= 1) & (aha17 <= 6)
    is_mid       = (aha17 >= 7) & (aha17 <= 12)
    in_zone      = (theta >= inf_theta_min) & (theta <= inf_theta_max)
    is_ant_lat   = (aha5 == 1) | (aha5 == 2)

    to_fix = is_basal_mid & in_zone & is_ant_lat

    new_aha17 = aha17.copy()
    new_aha5  = aha5.copy()
    new_aha17[to_fix & is_basal] = 6
    new_aha17[to_fix & is_mid]   = 12
    new_aha5[to_fix] = 3

    if verbose:
        n = int(to_fix.sum())
        print(f'Septal-Inferior boundary fix: reassigned {n} cells '
              f'from Anterior/Lateral to Inferior '
              f'(angular zone theta in [{inf_theta_min:+.0f}, {inf_theta_max:+.0f}] deg).')
        for orig_seg in (3, 4, 5, 9, 10, 11):
            m = to_fix & (aha17 == orig_seg)
            if m.any():
                print(f'  from AHA17 seg {orig_seg:2d}: {int(m.sum()):4d} cells')

    return new_aha17, new_aha5, int(to_fix.sum())




## Visualisation helpers


In [6]:
# ================================
# 4) Visualisation helpers
# ================================

def extract_surface(dataset):
    surf = dataset.extract_surface(algorithm='dataset_surface').clean()
    surf = surf.triangulate().clean()
    return surf


def extract_label_boundaries(surface, label):
    if label not in surface.cell_data:
        return None

    pieces = []
    for rid in np.unique(surface.cell_data[label]):
        ids = np.flatnonzero(surface.cell_data[label] == rid)
        if ids.size == 0:
            continue
        part = surface.extract_cells(ids)
        edges = part.extract_feature_edges(
            boundary_edges=True,
            non_manifold_edges=False,
            feature_edges=False,
            manifold_edges=False,
        )
        if edges.n_cells > 0:
            pieces.append(edges)

    if not pieces:
        return None

    out = pieces[0]
    for p in pieces[1:]:
        out = out.merge(p)
    return out.clean()


def compute_camera(bounds, view='oblique'):
    """Camera positions keyed off the mesh bounds.

    view='oblique'      : long-axis 3/4 view (anterior/front exterior)
    view='oblique_back' : 180° rotation of 'oblique' around the long axis (z),
                          giving the posterior/back exterior at the matching tilt.
    view='short_axis'   : looking down from above the base (septum ↔ lateral on the circle)
    """
    xmin, xmax, ymin, ymax, zmin, zmax = bounds
    center = np.array([(xmin + xmax) / 2, (ymin + ymax) / 2, (zmin + zmax) / 2])
    diag = np.linalg.norm([xmax - xmin, ymax - ymin, zmax - zmin])
    if view == 'short_axis':
        pos = center + np.array([0.0, 0.0, 1.9]) * diag
        up  = (0.0, 1.0, 0.0)
    elif view == 'oblique_back':
        # Mirror the 'oblique' offset through the long axis (z) so the
        # posterior wall faces the camera at the same tilt.
        pos = center + np.array([-1.35, 1.85, 1.05]) * diag
        up  = (0.0, 0.0, 1.0)
    else:
        pos = center + np.array([1.35, -1.85, 1.05]) * diag
        up  = (0.0, 0.0, 1.0)
    return [tuple(pos), tuple(center), up]


def add_edges(pl, edges, color, width, alpha=1.0):
    if edges is None or edges.n_cells == 0:
        return
    pl.add_mesh(edges, color=color, line_width=width, opacity=alpha, lighting=False)


def add_plain_surface(pl, mesh, color=DEFAULT_SURFACE_COLOR, opacity=1.0, show_wire=False):
    pl.add_mesh(
        mesh,
        color=color,
        smooth_shading=True,
        opacity=opacity,
        ambient=0.32,
        diffuse=0.70,
        specular=0.05,
        silhouette=dict(color=OUTLINE_COLOR, line_width=OUTLINE_WIDTH),
    )
    if show_wire:
        pl.add_mesh(
            mesh,
            style='wireframe',
            color=WIRE_COLOR,
            opacity=WIRE_ALPHA,
            line_width=WIRE_WIDTH,
            lighting=False,
        )


def add_categorical_surface(pl, mesh, scalars, cmap, clim, opacity=1.0):
    """Colour a surface by discrete cell labels.

    We pass a precomputed (N,3) RGB cell array via `rgb=True` to sidestep
    VTK's IndexedLookup, which remaps colours when some categories are
    absent from the current sub-mesh.
    """
    rgb_name = f'{scalars}_rgb'
    if rgb_name in mesh.cell_data:
        pl.add_mesh(
            mesh,
            scalars=rgb_name,
            preference='cell',
            rgb=True,
            show_scalar_bar=False,
            smooth_shading=True,
            opacity=opacity,
            ambient=0.32,
            diffuse=0.70,
            specular=0.05,
            silhouette=dict(color=OUTLINE_COLOR, line_width=OUTLINE_WIDTH),
        )
    else:
        pl.add_mesh(
            mesh,
            scalars=scalars,
            preference='cell',
            cmap=cmap,
            clim=clim,
            categories=True,
            show_scalar_bar=False,
            interpolate_before_map=False,
            smooth_shading=True,
            opacity=opacity,
            ambient=0.32,
            diffuse=0.70,
            specular=0.05,
            silhouette=dict(color=OUTLINE_COLOR, line_width=OUTLINE_WIDTH),
        )


def add_continuous_surface(pl, mesh, scalars, clim, opacity=1.0):
    pl.add_mesh(
        mesh,
        scalars=scalars,
        preference='cell',
        cmap=DIV_CMAP,
        clim=clim,
        show_scalar_bar=False,
        interpolate_before_map=True,
        smooth_shading=True,
        opacity=opacity,
        ambient=0.32,
        diffuse=0.70,
        specular=0.05,
        silhouette=dict(color=OUTLINE_COLOR, line_width=OUTLINE_WIDTH),
    )


def _render_panel(pl, dataset, draw_fn, title, camera):
    if PARALLEL_PROJ:
        pl.enable_parallel_projection()
    draw_fn(pl, dataset)
    pl.add_text(title, position='upper_left', font_size=14, color='black')
    pl.camera_position = camera
    pl.reset_camera()
    pl.camera.zoom(ZOOM)


def dual_view_image(dataset, draw_fn, panel_titles=('Anterior view', 'Posterior view'), window_size=WINDOW_SIZE):
    """Two oblique exterior views: anterior (front) and posterior (back).

    Replaces the original exterior + long-axis cutaway pair. Same dataset is
    rendered twice from cameras 180° apart around the long axis so the full
    epicardial surface (and any AHA segmentation on it) is visible across
    the two panels without a clipping operation.
    """
    cam_front = compute_camera(dataset.bounds, view='oblique')
    cam_back  = compute_camera(dataset.bounds, view='oblique_back')
    cameras = (cam_front, cam_back)

    pl = pv.Plotter(shape=(1, 2), off_screen=True, window_size=window_size, border=False)
    pl.set_background('white')
    for j in range(2):
        pl.subplot(0, j)
        _render_panel(pl, dataset, draw_fn, panel_titles[j], cameras[j])
    img = pl.screenshot(return_img=True)
    pl.close()
    return np.asarray(img)


def triple_view_image(dataset, draw_fn,
                      panel_titles=('Anterior view', 'Posterior view', 'Short-axis (basal)'),
                      window_size=WINDOW_SIZE_3):
    """Anterior exterior + posterior exterior + short-axis basal cap.

    The middle panel previously showed a long-axis cutaway; it has been
    replaced by the same dataset rendered from a posterior camera so the
    full epicardial AHA pattern is visible across the front/back pair.
    [P1] The short-axis panel still makes the septum-vs-lateral ring
    unambiguous, which neither oblique exterior view can.
    """
    center = np.array(dataset.center)
    bounds = dataset.bounds
    zmax = bounds[5]
    zmin = bounds[4]
    # Cut a short slab near the base so we see a ring of tissue from above.
    slab_hi = zmax - 0.08 * (zmax - zmin)
    slab_lo = zmax - 0.35 * (zmax - zmin)
    short_axis = dataset.clip(normal=(0, 0, 1), origin=(center[0], center[1], slab_lo), invert=False)
    short_axis = short_axis.clip(normal=(0, 0, -1), origin=(center[0], center[1], slab_hi), invert=False)

    cam_front = compute_camera(bounds, view='oblique')
    cam_back  = compute_camera(bounds, view='oblique_back')
    cam_short = compute_camera(bounds, view='short_axis')

    pl = pv.Plotter(shape=(1, 3), off_screen=True, window_size=window_size, border=False)
    pl.set_background('white')
    specs = [
        (dataset,    panel_titles[0], cam_front),
        (dataset,    panel_titles[1], cam_back),
        (short_axis, panel_titles[2], cam_short),
    ]
    for j, (ds, t, cam) in enumerate(specs):
        pl.subplot(0, j)
        _render_panel(pl, ds, draw_fn, t, cam)
    img = pl.screenshot(return_img=True)
    pl.close()
    return np.asarray(img)


def compose_basic(img, out_path, legend_handles=None, title=None, notes=None, colorbar=None):
    h, w = img.shape[:2]
    fig_w = 14
    fig_h = fig_w * h / w + 1.2
    footer_h = 1.0 if (legend_handles or notes or colorbar) else 0.0

    fig = plt.figure(figsize=(fig_w, fig_h + footer_h), facecolor='white')
    if footer_h > 0:
        gs = fig.add_gridspec(2, 1, height_ratios=[fig_h, footer_h], hspace=0.02)
        ax_img = fig.add_subplot(gs[0])
        ax_foot = fig.add_subplot(gs[1])
        ax_foot.axis('off')
    else:
        ax_img = fig.add_subplot(111)
        ax_foot = None

    ax_img.imshow(img)
    ax_img.axis('off')
    if title:
        fig.suptitle(title, fontsize=13, y=0.995)

    if ax_foot is not None:
        if legend_handles:
            ax_foot.legend(
                handles=legend_handles,
                loc='center left',
                ncol=max(1, min(4, len(legend_handles))),
                fontsize=9,
                frameon=False,
                bbox_to_anchor=(0.01, 0.55),
            )
        if notes:
            ax_foot.text(
                0.01, 0.05, notes,
                transform=ax_foot.transAxes,
                fontsize=9,
                color='#444444',
                va='bottom',
            )
        if colorbar is not None:
            # Right-aligned when a legend sits on the left, otherwise centred.
            # When notes are also present without a legend, push colorbar to the
            # right so the note text on the left is not overwritten.
            if legend_handles:
                x0 = 0.67
            elif notes:
                x0 = 0.55
            else:
                x0 = 0.35
            cax = ax_foot.inset_axes([x0, 0.35, 0.32, 0.28])
            norm = Normalize(vmin=colorbar['vmin'], vmax=colorbar['vmax'])
            cb = plt.colorbar(
                plt.cm.ScalarMappable(norm=norm, cmap=colorbar.get('cmap', DIV_CMAP)),
                cax=cax,
                orientation='horizontal',
            )
            cb.set_label(colorbar['label'], fontsize=9)
            cb.ax.tick_params(labelsize=8)

    out_path = Path(out_path)
    fig.savefig(out_path, dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.close(fig)   # [cleanup] was plt.show() — unused in off-screen runs, only emits a warning
    print(f'Saved: {out_path}')
    return out_path


def compute_robust_clim(grid, scalar_name, pct=STRAIN_CLIM_PCT, basal_exclude=STRAIN_BASAL_EXCLUDE):
    """Symmetric clim from a mass-of-wall percentile, excluding the basal rim.

    [P3] The exposed cut face at the base carries physically real but
    visually dominant extremes. We compute clim on cells whose centroid
    z-coordinate is below (1 - basal_exclude) of the z-extent.
    """
    vals = np.asarray(grid.cell_data[scalar_name])
    cc = grid.cell_centers().points
    zmin, zmax = cc[:, 2].min(), cc[:, 2].max()
    cutoff = zmax - basal_exclude * (zmax - zmin)
    keep = cc[:, 2] <= cutoff
    subset = vals[keep] if keep.any() else vals
    lo, hi = np.nanpercentile(subset, pct)
    vmax = max(abs(lo), abs(hi))
    if vmax == 0:
        vmax = 1.0
    return (-float(vmax), float(vmax))


def compute_sym_clim(values, pct=(2, 98), center_zero=True):
    """Legacy helper kept for any external caller. Prefer compute_robust_clim."""
    vals = np.asarray(values)
    lo, hi = np.nanpercentile(vals, pct)
    if center_zero:
        vmax = max(abs(lo), abs(hi))
        if vmax == 0:
            vmax = 1.0
        return (-float(vmax), float(vmax))
    return (float(lo), float(hi))



## Figure-generation functions


In [7]:
# ================================
# 5) Figure-generation functions
# ================================

def figure_geometry_plain(grid):
    def draw(pl, ds):
        surf = extract_surface(ds)
        add_plain_surface(pl, surf, show_wire=True)

    img = dual_view_image(grid, draw)
    handles = [
        Line2D([0], [0], color=OUTLINE_COLOR, lw=OUTLINE_WIDTH, label='LV outer silhouette'),
        Line2D([0], [0], color=WIRE_COLOR, lw=1.2, label='Surface wireframe'),
    ]
    return compose_basic(
        img,
        OUTPUT_DIR / 'fig_01_geometry_plain.png',
        legend_handles=handles,
        title='Left ventricle geometry (no colouring)',
        notes='Left: anterior (front) exterior view. Right: posterior (back) exterior view.',
    )


def figure_aha17_boundaries(grid):
    def draw(pl, ds):
        surf = extract_surface(ds)
        add_plain_surface(pl, surf, show_wire=False)
        add_edges(pl, extract_label_boundaries(surf, 'AHA17'),
                  AHA17_EDGE_COLOR, AHA17_EDGE_WIDTH, AHA17_EDGE_ALPHA)

    # [P0] Now visible because edge contrast is strong on the lightened surface.
    img = triple_view_image(
        grid, draw,
        panel_titles=('Anterior view', 'Posterior view', 'Short-axis (basal)'),
    )
    handles = [
        Line2D([0], [0], color=AHA17_EDGE_COLOR, lw=AHA17_EDGE_WIDTH + 0.2, label='AHA 17 boundaries'),
        Line2D([0], [0], color=OUTLINE_COLOR, lw=OUTLINE_WIDTH, label='LV outer silhouette'),
    ]
    return compose_basic(
        img,
        OUTPUT_DIR / 'fig_02_aha17_boundaries.png',
        legend_handles=handles,
        title='AHA 17 segmentation (neutral surface, no fill colouring)',
    )


def figure_aha5_grouped(grid):
    def draw(pl, ds):
        surf = extract_surface(ds)
        add_categorical_surface(pl, surf, 'AHA5', AHA5_CMAP, (1, 5))
        add_edges(pl, extract_label_boundaries(surf, 'AHA17'),
                  AHA17_EDGE_COLOR, AHA17_EDGE_WIDTH, AHA17_EDGE_ALPHA)
        add_edges(pl, extract_label_boundaries(surf, 'AHA5'),
                  AHA5_EDGE_COLOR, AHA5_EDGE_WIDTH, AHA5_EDGE_ALPHA)

    # [P1] Three panels: anterior + posterior epicardium + short-axis basal ring;
    # the short-axis panel still makes septum ↔ lateral unambiguous.
    img = triple_view_image(
        grid, draw,
        panel_titles=('Anterior view', 'Posterior view', 'Short-axis (basal)'),
    )
    handles = [mpatches.Patch(facecolor=AHA5_COLORS[i - 1], edgecolor='none',
                              label=AHA5_GROUP_NAMES[i]) for i in range(1, 6)]
    handles += [
        Line2D([0], [0], color=AHA17_EDGE_COLOR, lw=AHA17_EDGE_WIDTH + 0.2, label='AHA 17 boundaries'),
        Line2D([0], [0], color=AHA5_EDGE_COLOR, lw=AHA5_EDGE_WIDTH, label='AHA 5 boundaries'),
    ]
    return compose_basic(
        img,
        OUTPUT_DIR / 'fig_03_aha5_grouped.png',
        legend_handles=handles,
        title='AHA 5 grouped visualisation (AHA 17 + AHA 5 boundaries)',
    )


def figure_displacement_overlay(grid_ref, grid_def, dis):
    mean_disp = float(np.linalg.norm(dis, axis=1).mean())
    max_disp = float(np.linalg.norm(dis, axis=1).max())

    cam_front = compute_camera(grid_ref.bounds, view='oblique')
    cam_back  = compute_camera(grid_ref.bounds, view='oblique_back')

    pl = pv.Plotter(shape=(1, 2), off_screen=True, window_size=WINDOW_SIZE, border=False)
    pl.set_background('white')

    # Two oblique exterior overlays, anterior and posterior (180° around z).
    panel_specs = [
        ('Anterior overlay',  cam_front),
        ('Posterior overlay', cam_back),
    ]
    for j, (panel_title, camera) in enumerate(panel_specs):
        pl.subplot(0, j)
        if PARALLEL_PROJ:
            pl.enable_parallel_projection()
        s0 = extract_surface(grid_ref)
        s1 = extract_surface(grid_def)
        pl.add_mesh(s0, style='wireframe', color='#B0B0B0', line_width=0.8,
                    opacity=0.9, lighting=False)
        pl.add_mesh(
            s1,
            color='#666666',
            smooth_shading=True,
            opacity=0.85,
            ambient=0.30,
            diffuse=0.70,
            specular=0.05,
            silhouette=dict(color='#222222', line_width=2.0),
        )
        pl.add_text(panel_title, position='upper_left', font_size=14, color='black')
        pl.camera_position = camera
        pl.reset_camera()
        pl.camera.zoom(ZOOM)

    img = pl.screenshot(return_img=True)
    pl.close()

    handles = [
        Line2D([0], [0], color='#B0B0B0', lw=1.4, label='Original geometry (wireframe)'),
        Line2D([0], [0], color='#666666', lw=3.0, label='End-diastolic displaced geometry'),
    ]
    note = f'Mean nodal displacement = {mean_disp:.3f}; max nodal displacement = {max_disp:.3f}.'
    return compose_basic(
        img,
        OUTPUT_DIR / 'fig_04_ed_displacement.png',
        legend_handles=handles,
        title='Overall end-diastolic displacement (no scalar colouring)',
        notes=note,
    )


def figure_strain(grid, direction, grouped=False):
    scalar = f'strain_{direction}'
    clim = compute_robust_clim(grid, scalar)   # [P3]

    def draw(pl, ds):
        surf = extract_surface(ds)
        add_continuous_surface(pl, surf, scalar, clim)
        if grouped:
            add_edges(pl, extract_label_boundaries(surf, 'AHA17'),
                      AHA17_EDGE_COLOR, AHA17_EDGE_WIDTH, AHA17_EDGE_ALPHA)
            add_edges(pl, extract_label_boundaries(surf, 'AHA5'),
                      AHA5_EDGE_COLOR, AHA5_EDGE_WIDTH, AHA5_EDGE_ALPHA)

    img = dual_view_image(grid, draw)
    handles = None
    # The percentile rule for clim is mentioned in the cell comment; no on-figure
    # note so the footer (legend + colorbar) stays uncluttered.
    if grouped:
        handles = [
            Line2D([0], [0], color=AHA17_EDGE_COLOR, lw=AHA17_EDGE_WIDTH + 0.2, label='AHA 17 boundaries'),
            Line2D([0], [0], color=AHA5_EDGE_COLOR, lw=AHA5_EDGE_WIDTH, label='AHA 5 boundaries'),
        ]

    number_map = {
        ('circ', False): '05', ('rad', False): '06', ('lon', False): '07',
        ('circ', True):  '08', ('rad', True):  '09', ('lon', True):  '10',
    }
    suffix = 'grouped' if grouped else 'full'
    out_path = OUTPUT_DIR / f"fig_{number_map[(direction, grouped)]}_strain_{direction}_{suffix}.png"
    title = f"{STRAIN_LABELS[direction]} — {'with AHA 5 grouping' if grouped else 'full-field'}"
    return compose_basic(
        img,
        out_path,
        legend_handles=handles,
        title=title,
        colorbar={'vmin': clim[0], 'vmax': clim[1], 'label': STRAIN_LABELS[direction]},
    )


def figure_lateral_strain(grid, direction):
    idx = np.flatnonzero(np.isin(grid.cell_data['AHA17'], LATERAL_SEGMENTS))
    lat = grid.extract_cells(idx)
    scalar = f'strain_{direction}'
    clim = compute_robust_clim(grid, scalar)   # use full-grid clim so the 3 lateral figures are comparable

    def draw(pl, ds):
        surf = extract_surface(ds)
        add_continuous_surface(pl, surf, scalar, clim)
        add_edges(pl, extract_label_boundaries(surf, 'AHA17'), AHA17_EDGE_COLOR, 1.4, 0.98)

    img = dual_view_image(lat, draw, panel_titles=('Lateral (anterior view)', 'Lateral (posterior view)'))
    handles = [
        Line2D([0], [0], color=AHA17_EDGE_COLOR, lw=1.4, label='Lateral subsegment boundaries'),
    ]
    num_map = {'circ': '11', 'rad': '12', 'lon': '13'}
    out_path = OUTPUT_DIR / f"fig_{num_map[direction]}_lateral_{direction}.png"
    title = f"Lateral group subsegments — {STRAIN_LABELS[direction]}"
    return compose_basic(
        img,
        out_path,
        legend_handles=handles,
        title=title,
        colorbar={'vmin': clim[0], 'vmax': clim[1], 'label': STRAIN_LABELS[direction]},
    )


def extract_lateral_data(grid):
    aha17 = np.asarray(grid.cell_data['AHA17'])
    data = {}
    for seg in LATERAL_SEGMENTS:
        mask = aha17 == seg
        if not mask.any():
            raise ValueError(f'Segment {seg} has no cells.')
        data[seg] = {
            key: np.asarray(grid.cell_data[f'strain_{key}'])[mask]
            for key in STRAIN_COLS
        }
    return data


def plot_lateral_kde_grid(data):
    dirs = ['circ', 'rad', 'lon']
    segs = LATERAL_SEGMENTS
    fig, axes = plt.subplots(len(dirs), len(segs), figsize=(12.5, 7.8), sharex='row', facecolor='white')

    for i, direction in enumerate(dirs):
        all_vals = np.concatenate([data[s][direction] for s in segs])
        lo, hi = np.percentile(all_vals, [2.5, 97.5])
        pad = 0.05 * (hi - lo if hi > lo else 1.0)
        xx = np.linspace(lo - pad, hi + pad, 400)

        for j, seg in enumerate(segs):
            ax = axes[i, j]
            vals = data[seg][direction]
            color = LATERAL_SUB_COLORS[j]

            if len(vals) > 5 and np.std(vals) > 1e-10:
                kde = gaussian_kde(vals)
                yy = kde(xx)
                ax.fill_between(xx, yy, color=color, alpha=0.55)
                ax.plot(xx, yy, color=color, lw=1.25)

            ax.axvline(vals.mean(), color='#222222', ls='--', lw=0.9)
            ax.axvline(0, color='#999999', lw=0.7)
            ax.set_yticks([])
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_visible(False)

            if i == 0:
                ax.set_title(f'Segment {seg}', fontsize=11, color=color, fontweight='bold')
            if j == 0:
                ax.set_ylabel(STRAIN_LABELS[direction], fontsize=10)
            if i == len(dirs) - 1:
                ax.set_xlabel('strain value', fontsize=9)

            ax.text(
                0.98, 0.92,
                f"μ={vals.mean():+.3f}\nσ={vals.std():.3f}\nn={len(vals)}",
                transform=ax.transAxes, ha='right', va='top', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='#BBBBBB', alpha=0.9),
            )

    fig.suptitle('Lateral group: strain distributions across the four subsegments', fontsize=12)
    fig.tight_layout()
    out_path = OUTPUT_DIR / 'fig_14_lateral_distribution_kde.png'
    fig.savefig(out_path, dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'Saved: {out_path}')
    return out_path


def plot_lateral_boxplot(data):
    dirs = ['circ', 'rad', 'lon']
    segs = LATERAL_SEGMENTS
    fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.8), facecolor='white')

    for i, direction in enumerate(dirs):
        ax = axes[i]
        vals = [data[s][direction] for s in segs]
        bp = ax.boxplot(vals, patch_artist=True, widths=0.6, showfliers=False)

        for patch, color in zip(bp['boxes'], LATERAL_SUB_COLORS):
            patch.set(facecolor=color, alpha=0.55, edgecolor='#444444')
        for med in bp['medians']:
            med.set(color='#111111', linewidth=1.2)

        ax.axhline(0, color='#999999', lw=0.8)
        ax.set_xticks(range(1, len(segs) + 1), [str(s) for s in segs])
        ax.set_title(STRAIN_LABELS[direction], fontsize=11)
        ax.set_xlabel('Lateral subsegment (AHA17 label)', fontsize=9)
        if i == 0:
            ax.set_ylabel('strain value', fontsize=9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    legend_handles = [
        mpatches.Patch(facecolor=LATERAL_SUB_COLORS[i], alpha=0.55, edgecolor='#444444', label=f'Segment {segs[i]}')
        for i in range(len(segs))
    ]
    fig.legend(handles=legend_handles, loc='upper center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.03))
    fig.suptitle('Lateral group: boxplots for the four subsegments', fontsize=12, y=1.08)
    fig.tight_layout()
    out_path = OUTPUT_DIR / 'fig_15_lateral_distribution_boxplot.png'
    fig.savefig(out_path, dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'Saved: {out_path}')
    return out_path



## Load data and construct the grids

- `grid_ref`: original / reference geometry
- `grid_ed`: displaced geometry using `node_ori + dis`


In [8]:
# ================================
# 6) Load all data and build grids
# ================================

points, conn = parse_abaqus_inp(INP_PATH)
n_points = points.shape[0]
n_cells = conn.shape[0]
print(f'Mesh: {n_points} points, {n_cells} cells')

node_regions, aha17, aha5, conflict_mask = load_aha_labels(AHA_MAT, n_cells)
print(f'Conflict cells reassigned to segment 17: {int(conflict_mask.sum())}')

# Data-cleaning step: Ant/Lat labels at the Septal-Inferior boundary are wrong.
# See fix_septal_inferior_boundary() docstring. The angular thresholds below
# are tunable; [+30, +90] degrees matches the observed Inferior zone plus a
# small buffer toward InfSept. Cell centres are computed directly from the
# Abaqus nodes and connectivity (do not need the PyVista grid yet).
_cell_centers_tmp = points[conn].mean(axis=1)
aha17, aha5, _n_fixed = fix_septal_inferior_boundary(
    aha17, aha5, _cell_centers_tmp,
    inf_theta_min=30.0, inf_theta_max=90.0,
)
del _cell_centers_tmp

S = load_strain_struct(STRAIN_MAT, n_cells_expected=n_cells, n_points_expected=n_points)
print(f'strain_tensor_cra shape: {S["strain_tensor"].shape}')
print(f'dis shape: {S["dis"].shape}')

max_abs_point_diff = float(np.abs(points - S['node_ori']).max())
print(f'Max |points - node_ori| = {max_abs_point_diff:.6e}')

# [P2] Print the three columns we map to circ/rad/lon so the convention is auditable.
print('\nStrain-tensor column contract (edit STRAIN_COLS in the config cell if wrong):')
for key in ('circ', 'rad', 'lon'):
    col = S['strain_tensor'][:, STRAIN_COLS[key]]
    print(f"  {key:>5s} <- col {STRAIN_COLS[key]}:  mean={col.mean():+.4f}  median={np.median(col):+.4f}  "
          f"std={col.std():.4f}  range=[{col.min():+.4f}, {col.max():+.4f}]")
print('Expected at ED vs unloaded reference: circ +, rad -, lon + (all ~0.1-0.2 in magnitude).')

grid_ref = build_grid(points, conn, node_regions, aha17, aha5, S['strain_tensor'], S['dis'])
grid_ed  = build_grid(points + S['dis'], conn, node_regions, aha17, aha5, S['strain_tensor'], S['dis'])

if SAVE_VTU:
    ref_vtu = OUTPUT_DIR / 'lv_reference_with_labels_and_strain.vtu'
    ed_vtu  = OUTPUT_DIR / 'lv_end_diastolic_with_labels_and_strain.vtu'
    grid_ref.save(ref_vtu)
    grid_ed.save(ed_vtu)
    print(f'Saved VTU: {ref_vtu}')
    print(f'Saved VTU: {ed_vtu}')

print('\nAHA5 cell counts:')
for gid in sorted(AHA5_GROUP_NAMES):
    print(f'  {gid} - {AHA5_GROUP_NAMES[gid]:9s}: {int((aha5 == gid).sum())}')

print('\nLateral subsegments used in this notebook:', LATERAL_SEGMENTS)


Mesh: 31856 points, 28650 cells
Conflict cells reassigned to segment 17: 600
Septal-Inferior boundary fix: reassigned 252 cells from Anterior/Lateral to Inferior (angular zone theta in [+30, +90] deg).
  from AHA17 seg  4:   60 cells
  from AHA17 seg  5:   10 cells
  from AHA17 seg 10:  140 cells
  from AHA17 seg 11:   42 cells
strain_tensor_cra shape: (28650, 6)
dis shape: (31856, 3)
Max |points - node_ori| = 4.999948e-07

Strain-tensor column contract (edit STRAIN_COLS in the config cell if wrong):
   circ <- col 0:  mean=+0.1189  median=+0.1113  std=0.0512  range=[-0.0070, +0.4240]
    rad <- col 1:  mean=-0.1310  median=-0.1316  std=0.0576  range=[-0.3003, +0.0512]
    lon <- col 2:  mean=+0.1555  median=+0.1515  std=0.0397  range=[-0.0404, +0.4279]
Expected at ED vs unloaded reference: circ +, rad -, lon + (all ~0.1-0.2 in magnitude).
Saved VTU: merged_lv_visualization_outputs\lv_reference_with_labels_and_strain.vtu
Saved VTU: merged_lv_visualization_outputs\lv_end_diastolic_with_


## Generate geometry / segmentation / displacement figures


In [9]:

# ================================
# 7) Geometry, segmentation, displacement
# ================================

figure_geometry_plain(grid_ref)
figure_aha17_boundaries(grid_ref)
figure_aha5_grouped(grid_ref)
figure_displacement_overlay(grid_ref, grid_ed, S['dis'])


Saved: merged_lv_visualization_outputs\fig_01_geometry_plain.png
Saved: merged_lv_visualization_outputs\fig_02_aha17_boundaries.png
Saved: merged_lv_visualization_outputs\fig_03_aha5_grouped.png
Saved: merged_lv_visualization_outputs\fig_04_ed_displacement.png


WindowsPath('merged_lv_visualization_outputs/fig_04_ed_displacement.png')


## Generate full-field strain figures


In [10]:

# ================================
# 8) Full-field strain figures
# ================================

for direction in ['circ', 'rad', 'lon']:
    figure_strain(grid_ref, direction, grouped=False)


Saved: merged_lv_visualization_outputs\fig_05_strain_circ_full.png
Saved: merged_lv_visualization_outputs\fig_06_strain_rad_full.png
Saved: merged_lv_visualization_outputs\fig_07_strain_lon_full.png



## Generate AHA5-grouped strain figures


In [11]:

# ================================
# 9) AHA5-grouped strain figures
# ================================

for direction in ['circ', 'rad', 'lon']:
    figure_strain(grid_ref, direction, grouped=True)


Saved: merged_lv_visualization_outputs\fig_08_strain_circ_grouped.png
Saved: merged_lv_visualization_outputs\fig_09_strain_rad_grouped.png
Saved: merged_lv_visualization_outputs\fig_10_strain_lon_grouped.png



## Generate Lateral-group subsegment strain figures


In [12]:

# ================================
# 10) Lateral-group subsegment strain figures
# ================================

for direction in ['circ', 'rad', 'lon']:
    figure_lateral_strain(grid_ref, direction)


Saved: merged_lv_visualization_outputs\fig_11_lateral_circ.png
Saved: merged_lv_visualization_outputs\fig_12_lateral_rad.png
Saved: merged_lv_visualization_outputs\fig_13_lateral_lon.png



## Generate Lateral-group strain distribution plots


In [13]:

# ================================
# 11) Lateral-group strain distributions
# ================================

lateral_data = extract_lateral_data(grid_ref)
plot_lateral_kde_grid(lateral_data)
plot_lateral_boxplot(lateral_data)


Saved: merged_lv_visualization_outputs\fig_14_lateral_distribution_kde.png
Saved: merged_lv_visualization_outputs\fig_15_lateral_distribution_boxplot.png


WindowsPath('merged_lv_visualization_outputs/fig_15_lateral_distribution_boxplot.png')


## Output summary


In [14]:

# ================================
# 12) List output files
# ================================

output_files = sorted(OUTPUT_DIR.glob('*'))
for p in output_files:
    print(p)


merged_lv_visualization_outputs\fig_01_geometry_plain.png
merged_lv_visualization_outputs\fig_02_aha17_boundaries.png
merged_lv_visualization_outputs\fig_03_aha5_grouped.png
merged_lv_visualization_outputs\fig_04_ed_displacement.png
merged_lv_visualization_outputs\fig_05_strain_circ_full.png
merged_lv_visualization_outputs\fig_06_strain_rad_full.png
merged_lv_visualization_outputs\fig_07_strain_lon_full.png
merged_lv_visualization_outputs\fig_08_strain_circ_grouped.png
merged_lv_visualization_outputs\fig_09_strain_rad_grouped.png
merged_lv_visualization_outputs\fig_10_strain_lon_grouped.png
merged_lv_visualization_outputs\fig_11_lateral_circ.png
merged_lv_visualization_outputs\fig_12_lateral_rad.png
merged_lv_visualization_outputs\fig_13_lateral_lon.png
merged_lv_visualization_outputs\fig_14_lateral_distribution_kde.png
merged_lv_visualization_outputs\fig_15_lateral_distribution_boxplot.png
merged_lv_visualization_outputs\lv_end_diastolic_with_labels_and_strain.vtu
merged_lv_visualizat